In [1]:
# =========================================================
# Setup
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchvision.models import resnet18
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import random
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)

print("Device:", device)

Device: cuda


In [2]:
# =========================================================
# Data
# =========================================================

class TwoCropsTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform
    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)

ssl_base_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.4,0.4,0.4,0.1),
    T.RandomGrayscale(p=0.2),
    T.ToTensor()
])

ssl_transform = TwoCropsTransform(ssl_base_transform)

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=ssl_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=T.ToTensor()
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:02<00:00, 57.3MB/s] 


In [3]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone
    def forward(self, x):
        return self.backbone(x)

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            nn.Linear(2048, 256)
        )
    def forward(self, x):
        return self.net(x)

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.projector = MLP()
    def forward(self, x):
        z = self.projector(self.encoder(x))
        return F.normalize(z, dim=1)

In [5]:
def nt_xent_loss(z1, z2, tau):
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)

    sim = torch.mm(z, z.t()) / tau
    mask = torch.eye(2*N, device=z.device).bool()
    sim.masked_fill_(mask, -9e15)

    positives = torch.cat([
        torch.arange(N, 2*N),
        torch.arange(0, N)
    ]).to(z.device)

    return F.cross_entropy(sim, positives)

In [6]:
def train_simclr_tau(model, tau, epochs=100):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    total_start = time.time()

    for epoch in range(epochs):

        epoch_start = time.time()
        model.train()
        total_loss = 0

        for (x1, x2), _ in train_loader:
            x1, x2 = x1.to(device), x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1, z2, tau)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        epoch_time = (time.time() - epoch_start)/60
        cumulative_time = (time.time() - total_start)/60

        print(f"[Tau={tau}] Epoch {epoch+1:03d}/100 | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Epoch Time: {epoch_time:.2f}m | "
              f"Total Time: {cumulative_time:.2f}m")

    return model.encoder

In [7]:
def jacobian_frobenius(model, x):
    x = x.clone().detach().to(device).requires_grad_(True)
    y = model(x)
    v = torch.randn_like(y)

    Jv = torch.autograd.grad(
        outputs=y,
        inputs=x,
        grad_outputs=v,
        retain_graph=False
    )[0]

    return torch.sqrt(Jv.pow(2).sum() / x.size(0))


def evaluate_encoder(encoder):

    encoder = encoder.to(device)
    encoder.eval()

    jac_vals = []
    noise_vals = []

    for i, (x, _) in enumerate(test_loader):
        if i == 5:
            break

        x = x.to(device)

        jac_vals.append(jacobian_frobenius(encoder, x).item())

        noise = torch.randn_like(x) * 0.1
        with torch.no_grad():
            f1 = encoder(x)
            f2 = encoder(x + noise)
            noise_vals.append(torch.norm(f1 - f2, dim=1).mean().item())

    return np.mean(jac_vals), np.mean(noise_vals)

In [8]:
taus = [0.1, 0.5, 1.0]
seeds = [8, 9]

all_results = []
global_start = time.time()

for tau in taus:
    for seed in seeds:

        print("\n==============================")
        print(f"Tau={tau} | Seed={seed}")
        print("==============================")

        set_seed(seed)
        seed_start = time.time()

        encoder = train_simclr_tau(SimCLR(), tau=tau, epochs=100)

        jac, noise = evaluate_encoder(encoder)

        all_results.append({
            "Tau": tau,
            "Seed": seed,
            "Jacobian": jac,
            "Noise_Sensitivity": noise
        })

        seed_time = (time.time() - seed_start)/60
        print(f"Finished Tau={tau}, Seed={seed} in {seed_time:.2f} minutes")

total_time = (time.time() - global_start)/60
print(f"\nTotal Experiment Time: {total_time:.2f} minutes")


Tau=0.1 | Seed=8
[Tau=0.1] Epoch 001/100 | Loss: 3.8841 | Epoch Time: 0.97m | Total Time: 0.97m
[Tau=0.1] Epoch 002/100 | Loss: 2.9381 | Epoch Time: 0.94m | Total Time: 1.91m
[Tau=0.1] Epoch 003/100 | Loss: 2.5786 | Epoch Time: 0.94m | Total Time: 2.85m
[Tau=0.1] Epoch 004/100 | Loss: 2.3372 | Epoch Time: 0.94m | Total Time: 3.79m
[Tau=0.1] Epoch 005/100 | Loss: 2.1530 | Epoch Time: 0.95m | Total Time: 4.74m
[Tau=0.1] Epoch 006/100 | Loss: 2.0141 | Epoch Time: 0.95m | Total Time: 5.69m
[Tau=0.1] Epoch 007/100 | Loss: 1.8954 | Epoch Time: 0.94m | Total Time: 6.63m
[Tau=0.1] Epoch 008/100 | Loss: 1.8038 | Epoch Time: 0.95m | Total Time: 7.58m
[Tau=0.1] Epoch 009/100 | Loss: 1.7036 | Epoch Time: 0.96m | Total Time: 8.54m
[Tau=0.1] Epoch 010/100 | Loss: 1.6587 | Epoch Time: 0.97m | Total Time: 9.52m
[Tau=0.1] Epoch 011/100 | Loss: 1.5797 | Epoch Time: 0.97m | Total Time: 10.49m
[Tau=0.1] Epoch 012/100 | Loss: 1.5290 | Epoch Time: 0.98m | Total Time: 11.47m
[Tau=0.1] Epoch 013/100 | Loss: 

In [9]:
results_df = pd.DataFrame(all_results)

print("\nRaw Results")
print(results_df)

summary = results_df.groupby("Tau").agg(
    Jacobian_Mean=("Jacobian","mean"),
    Jacobian_Std=("Jacobian","std"),
    Noise_Mean=("Noise_Sensitivity","mean"),
    Noise_Std=("Noise_Sensitivity","std")
)

print("\nSummary (Mean ± Std)")
print(summary)


Raw Results
   Tau  Seed    Jacobian  Noise_Sensitivity
0  0.1     8  106.107857          19.689766
1  0.1     9  105.366417          19.240178
2  0.5     8   87.666386          18.892843
3  0.5     9   88.850436          18.688172
4  1.0     8   78.463168          17.879443
5  1.0     9   75.776111          17.218259

Summary (Mean ± Std)
     Jacobian_Mean  Jacobian_Std  Noise_Mean  Noise_Std
Tau                                                    
0.1     105.737137      0.524277   19.464972   0.317906
0.5      88.258411      0.837250   18.790508   0.144724
1.0      77.119640      1.900037   17.548851   0.467528


In [10]:
results_df.to_csv("/kaggle/working/results.csv", index=False)